# Assignment 2 - Kuning Liang_1936708



# Preprocessing

In [3]:
import os, csv, math, random
from typing import List, Dict, Tuple

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

SEED = 1936708
random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)

TRAIN_CSV = "../data/train_data.csv"
TEST_CSV  = "../data/test_data.csv"

FEATURE_NAMES = [
    "Pregnancies", "Glucose", "BloodPressure", "SkinThickness",
    "Insulin", "BMI", "DiabetesPedigreeFunction", "Age"
]
TARGET_NAME = "Outcome"

VAL_RATIO  = 0.2
BATCH_SIZE = 32

ZERO_AS_MISSING = {"Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"}

Device: cpu


In [4]:
def read_csv(path: str) -> List[Dict[str, str]]:
    rows = []
    with open(path, newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        header = reader.fieldnames or []
        required = set(FEATURE_NAMES + [TARGET_NAME])
        assert required.issubset(set(header)), f"CSV is missing necessary columns: {required - set(header)}"
        for row in reader:
            rows.append(row)
    return rows

train_rows = read_csv(TRAIN_CSV)
test_rows  = read_csv(TEST_CSV)
print(f"train rows: {len(train_rows)}, test rows: {len(test_rows)}")

train rows: 614, test rows: 154


In [5]:
def to_float(x):
    try: return float(x)
    except: return float('nan')

def extract_matrix(rows: List[Dict[str,str]]):
    X, y = [], []
    for r in rows:
        feat = [to_float(r[k]) for k in FEATURE_NAMES]
        yval = int(float(r[TARGET_NAME]))
        X.append(feat); y.append(yval)
    return X, y

X_train_raw, y_train = extract_matrix(train_rows)
X_test_raw,  y_test  = extract_matrix(test_rows)

def column(values_2d, j):
    return [row[j] for row in values_2d]

def median_nonzero(xs):
    nz = [x for x in xs if (not math.isnan(x)) and x!=0.0]
    if not nz: return 0.0
    nz.sort()
    n = len(nz)
    return nz[n//2] if n%2==1 else (nz[n//2-1]+nz[n//2])/2

medians = {}
for j, name in enumerate(FEATURE_NAMES):
    col_tr = column(X_train_raw, j)
    if name in ZERO_AS_MISSING:
        med = median_nonzero(col_tr)
    else:
        v = [x for x in col_tr if not math.isnan(x)]
        v.sort(); n = len(v)
        med = v[n//2] if n%2==1 else (v[n//2-1]+v[n//2])/2
    medians[name] = med

def impute_rows(X):
    X2 = []
    for row in X:
        r2 = []
        for j, name in enumerate(FEATURE_NAMES):
            x = row[j]
            if math.isnan(x) or (name in ZERO_AS_MISSING and x==0.0):
                r2.append(medians[name])
            else:
                r2.append(x)
        X2.append(r2)
    return X2

X_train_imp = impute_rows(X_train_raw)
X_test_imp  = impute_rows(X_test_raw)

means, stds = {}, {}
for j, name in enumerate(FEATURE_NAMES):
    col = column(X_train_imp, j)
    mu  = sum(col)/len(col)
    var = sum((v-mu)**2 for v in col)/max(1,(len(col)-1))
    sigma = math.sqrt(var) if var>0 else 1.0
    means[name], stds[name] = mu, sigma

def zscore(X):
    Z = []
    for row in X:
        Z.append([(row[j]-means[FEATURE_NAMES[j]])/stds[FEATURE_NAMES[j]]
                  for j in range(len(FEATURE_NAMES))])
    return Z

X_train = zscore(X_train_imp)
X_test  = zscore(X_test_imp)

idx = list(range(len(X_train)))
random.shuffle(idx)
val_n  = int(len(idx)*VAL_RATIO)
val_id = set(idx[:val_n])
tr_id  = [i for i in idx if i not in val_id]

X_tr = [X_train[i] for i in tr_id]; y_tr = [y_train[i] for i in tr_id]
X_va = [X_train[i] for i in val_id]; y_va = [y_train[i] for i in val_id]

print(f"Split -> train: {len(X_tr)}, val: {len(X_va)}, test: {len(X_test)}")

Split -> train: 492, val: 122, test: 154


In [6]:
class TabularDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self): return self.X.size(0)
    def __getitem__(self, i): return self.X[i], self.y[i]

ds_tr = TabularDataset(X_tr, y_tr)
ds_va = TabularDataset(X_va, y_va)
ds_te = TabularDataset(X_test, y_test)

loader_tr = DataLoader(ds_tr, batch_size=BATCH_SIZE, shuffle=True)
loader_va = DataLoader(ds_va, batch_size=BATCH_SIZE, shuffle=False)
loader_te = DataLoader(ds_te, batch_size=BATCH_SIZE, shuffle=False)

print("Shapes ->", ds_tr.X.shape, ds_va.X.shape, ds_te.X.shape)
print("Train class balance:", {k: y_tr.count(k) for k in sorted(set(y_tr))})

Shapes -> torch.Size([492, 8]) torch.Size([122, 8]) torch.Size([154, 8])
Train class balance: {0: 326, 1: 166}


# Model Implementation

In [8]:
class MLP(nn.Module):
    def __init__(self, in_dim: int, num_classes: int, hidden=(32,)):
        super().__init__()
        layers = []
        last = in_dim
        for h in hidden:
            layers += [nn.Linear(last, h), nn.ReLU(inplace=True)]
            last = h
        layers += [nn.Linear(last, num_classes)]
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)

IN_DIM = len(FEATURE_NAMES)
NUM_CLASSES = 2
criterion = nn.CrossEntropyLoss()
print("IN_DIM:", IN_DIM, "| NUM_CLASSES:", NUM_CLASSES)

IN_DIM: 8 | NUM_CLASSES: 2


In [9]:
def evaluate(model, loader):
    model.eval()
    total_loss, total_correct, total = 0.0, 0, 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            total_loss   += float(loss.item()) * yb.size(0)
            total_correct+= int((logits.argmax(1)==yb).sum().item())
            total += yb.size(0)
    return total_loss/max(total,1), total_correct/max(total,1)

def train_one_epoch(model, loader, optimizer):
    model.train()
    total_loss, total_correct, total = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss   += float(loss.item()) * yb.size(0)
        total_correct+= int((logits.argmax(1)==yb).sum().item())
        total += yb.size(0)
    return total_loss/max(total,1), total_correct/max(total,1)

# Experiments

In [11]:
param_sets = [
    {"hidden": (32,),            "lr": 1e-3, "epochs": 30},
    {"hidden": (64, 32),         "lr": 5e-4, "epochs": 40},
    {"hidden": (128, 64, 32),    "lr": 1e-3, "epochs": 50},
]

best = {"val_acc": -1.0, "state": None, "cfg": None}
history = []

for i, cfg in enumerate(param_sets, 1):
    print(f"\n=== Param Set {i}: {cfg} ===")
    model = MLP(IN_DIM, NUM_CLASSES, hidden=cfg["hidden"]).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg["lr"])

    for ep in range(1, cfg["epochs"]+1):
        tr_loss, tr_acc = train_one_epoch(model, loader_tr, optimizer)
        va_loss, va_acc = evaluate(model, loader_va)
        if ep % 5 == 0 or ep == 1 or ep == cfg["epochs"]:
            print(f"[Set{i}] Epoch {ep:02d} | "
                  f"train loss {tr_loss:.4f} acc {tr_acc:.3f} | "
                  f"val loss {va_loss:.4f} acc {va_acc:.3f}")

    history.append({"set": i, **cfg, "val_acc": va_acc, "val_loss": va_loss})
    if va_acc > best["val_acc"]:
        best["val_acc"] = va_acc
        best["state"]   = model.state_dict()
        best["cfg"]     = cfg

print("\nSummary:", history)
print("Best:", best["cfg"], "ValAcc:", best["val_acc"])


=== Param Set 1: {'hidden': (32,), 'lr': 0.001, 'epochs': 30} ===
[Set1] Epoch 01 | train loss 0.6322 acc 0.665 | val loss 0.6093 acc 0.656
[Set1] Epoch 05 | train loss 0.5167 acc 0.744 | val loss 0.5155 acc 0.770
[Set1] Epoch 10 | train loss 0.4647 acc 0.783 | val loss 0.4881 acc 0.762
[Set1] Epoch 15 | train loss 0.4476 acc 0.787 | val loss 0.4879 acc 0.762
[Set1] Epoch 20 | train loss 0.4381 acc 0.797 | val loss 0.4905 acc 0.762
[Set1] Epoch 25 | train loss 0.4311 acc 0.797 | val loss 0.4919 acc 0.762
[Set1] Epoch 30 | train loss 0.4254 acc 0.799 | val loss 0.4926 acc 0.746

=== Param Set 2: {'hidden': (64, 32), 'lr': 0.0005, 'epochs': 40} ===
[Set2] Epoch 01 | train loss 0.6519 acc 0.663 | val loss 0.6453 acc 0.639
[Set2] Epoch 05 | train loss 0.5425 acc 0.703 | val loss 0.5461 acc 0.680
[Set2] Epoch 10 | train loss 0.4501 acc 0.783 | val loss 0.4933 acc 0.762
[Set2] Epoch 15 | train loss 0.4269 acc 0.795 | val loss 0.4921 acc 0.762
[Set2] Epoch 20 | train loss 0.4147 acc 0.803 | 

In [12]:
best_model = MLP(IN_DIM, NUM_CLASSES, hidden=best["cfg"]["hidden"]).to(device)
best_model.load_state_dict(best["state"])
test_loss, test_acc = evaluate(best_model, loader_te)
print(f"[TEST] loss {test_loss:.4f} acc {test_acc:.3f}")

torch.save(best_model.state_dict(), "best_mlp_diabetes.pth")

best_model.eval()
with torch.no_grad():
    for xb, yb in loader_te:
        preds = best_model(xb.to(device)).argmax(1).cpu().tolist()
        print("[Sample preds vs truth] (first 20):")
        for k in range(min(20, len(preds))):
            print(f"  #{k}: pred={preds[k]}, true={int(yb[k])}")
        break

[TEST] loss 0.4644 acc 0.740
[Sample preds vs truth] (first 20):
  #0: pred=0, true=0
  #1: pred=1, true=1
  #2: pred=1, true=1
  #3: pred=1, true=1
  #4: pred=0, true=0
  #5: pred=0, true=0
  #6: pred=0, true=0
  #7: pred=0, true=0
  #8: pred=1, true=0
  #9: pred=0, true=0
  #10: pred=1, true=0
  #11: pred=0, true=0
  #12: pred=0, true=0
  #13: pred=1, true=0
  #14: pred=0, true=0
  #15: pred=0, true=1
  #16: pred=0, true=0
  #17: pred=0, true=0
  #18: pred=0, true=0
  #19: pred=0, true=0
